In [ ]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset
import random
import time
import os
import torch.nn as nn
from tqdm import tqdm


In [ ]:
device = torch.device("cpu")
torch.backends.quantized.engine = 'qnnpack'

In [ ]:
model_1 = torch.jit.load("fault_free_model.pt")
model_1 = model_1.to(device)
print("✅")

In [ ]:
model_2 = torch.jit.load ("faulty_model.pt")
model_2.eval()
model_2 = model_2.to(device)
print("✅")

In [ ]:
model_3 = torch.jit.load("fault_free_model.pt")

model_3.eval()
model_3 = model_3.to(device)
print("✅")

In [ ]:
# dataset_path = "LC25000 dataset"
# dataset_path = "MRI dataset"
dataset_path = "Chest X-Ray dataset"
# test_dir = os.path.join(dataset_path, "test")

# test_transform = transforms.Compose([
#     transforms.Resize((224, 224)),
#     transforms.ToTensor(),
#     transforms.Normalize(mean=[0.485, 0.456, 0.406],
#                          std=[0.229, 0.224, 0.225])
# ])

test_transform = transforms.Compose([ # for the grayscale (Xray / MRI) datasets
    transforms.Lambda(lambda img: img.convert("RGB")),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
])


In [ ]:
test_dataset = datasets.ImageFolder(test_dir, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


# Design File saving

## DMR design

In [ ]:
import torch
import torch.nn as nn

class DMRScript(nn.Module):
    def __init__(self, m1, m2):
        super().__init__()
        self.m1 = m1
        self.m2 = m2

    def forward(self, x):
        o1 = self.m1(x)
        o2 = self.m2(x)

        p1 = torch.sigmoid(o1)
        p2 = torch.sigmoid(o2)

        b1 = p1 > 0.5
        b2 = p2 > 0.5

        # Agreement and disagreement
        agreement = (b1 == b2)
        disagreement = (b1 != b2)

        return {
            "b1": b1.to(torch.int64),
            "b2": b2.to(torch.int64),
            "agreement": agreement.to(torch.int64),
            "disagreement": disagreement.to(torch.int64)
        }

# Example usage:
dmr_model = DMRScript(model_1, model_2)
dmr_model.eval()

scripted_dmr = torch.jit.script(dmr_model)
scripted_dmr.save("DMR_model.pt")

device = "cpu"
DMR_model = torch.jit.load("DMR_model.pt", map_location=device)
DMR_model.eval()
print("DMR model ready")

In [ ]:
def evaluate_model(model, dataloader, device="cpu", N_list=[1, 10, 20, 50, 100, 500, 1000]):
    model.to(device)
    model.eval()

    results = []

    for N in N_list:
        total_samples = 0
        total_correct_agreement = 0
        total_agreements = 0
        total_disagreements = 0

        start_time = time.time()

        for inputs, labels in tqdm(dataloader):
            batch_size = inputs.size(0)

            # Stop if we have processed enough samples
            if total_samples >= N:
                break

            # Trim batch if it would exceed N
            if total_samples + batch_size > N:
                trim = N - total_samples
                inputs = inputs[:trim]
                labels = labels[:trim]
                batch_size = trim

            inputs = inputs.to(device)
            labels = labels.to(device)

            with torch.no_grad():
                y_pred = model(inputs)

                b1 = y_pred["b1"].view(-1)
                agreement = y_pred["agreement"].view(-1)

                agreements_count = agreement.sum().item()
                disagreements_count = (agreement == 0).sum().item()

                b1_agreed = b1[agreement == 1]
                labels_agreed = labels[agreement == 1]
                correct_agreement = (b1_agreed == labels_agreed).sum().item()

                total_correct_agreement += correct_agreement
                total_agreements += agreements_count
                total_disagreements += disagreements_count
                total_samples += batch_size

        elapsed_time = time.time() - start_time

# DMR disagreements are treated as unresolved outcomes.
# Therefore, accuracy is computed over all evaluated samples,
# not only the agreement cases.

accuracy = total_correct_agreement / total_samples * 100
        accuracy = total_correct_agreement / total_samples * 100

        print(f"\n✅ N = {total_samples} images → Accuracy: {accuracy:.2f}%, Time: {elapsed_time:.2f}s, Agreements: {total_agreements}, Disagreements: {total_disagreements}")

        results.append((accuracy, total_agreements, total_disagreements))

    return results

In [ ]:
N_list = [1, 10, 20, 50, 100, 400, 1000]
results = evaluate_model(DMR_model, test_loader, device="cpu", N_list=N_list)

## TMR design

In [ ]:
class TMRScript(nn.Module):
    def __init__(self, m1, m2, m3):
        super().__init__()
        self.m1 = m1
        self.m2 = m2
        self.m3 = m3

    def forward(self, x):

        o1 = self.m1(x)
        o2 = self.m2(x)
        o3 = self.m3(x)


        p1 = torch.sigmoid(o1)
        p2 = torch.sigmoid(o2)
        p3 = torch.sigmoid(o3)

        b1 = p1 > 0.5
        b2 = p2 > 0.5
        b3 = p3 > 0.5

        # majority vote
        tmr = (b1 & b2) | (b1 & b3) | (b2 & b3)

        return tmr.to(torch.int64)

In [ ]:
tmr_model = TMRScript(model_1, model_2, model_3)
tmr_model.eval()

scripted_tmr = torch.jit.script(tmr_model)
scripted_tmr.save("TMR_model.pt")

In [ ]:
device = "cpu"
TMR_model = torch.jit.load("TMR_model.pt", map_location=device)
TMR_model.eval()
print ("✅") #Just to make sure the .pt file can be loaded

In [ ]:
def evaluate_model_Deployment(model, dataloader, device="cpu", N_list=None):

    model.eval()
    model.to(device)

    if N_list is None:
        N_list = [len(dataloader.dataset)]

    for N in N_list:
        correct = 0
        total = 0
        num_processed = 0

        start_time = time.time()

        with torch.no_grad():
            for images, labels in tqdm(dataloader):
                if N is not None and num_processed >= N:
                    break

                images, labels = images.to(device), labels.to(device)

                if images.dim() == 3:
                    images = images.unsqueeze(0)

                outputs = model(images)

                predicted = outputs.view(-1)
                target_labels = labels.int().view(-1)

                correct += (predicted == target_labels).sum().item()
                total += target_labels.numel()
                num_processed += target_labels.numel()

        end_time = time.time()
        accuracy = 100 * correct / total
        elapsed_time = end_time - start_time

        print(f"\n N = {N} images → Accuracy: {accuracy:.2f}%, Time: {elapsed_time:.2f}s")

In [ ]:
N_list = [1, 10, 20, 50, 100, 400, 1000]
evaluate_model_Deployment(TMR_model, test_loader,device = "cpu", N_list=N_list)

# for the hybrid design, we used two type of code



## Heybrid design

### Hybrid Redundancy Implementations

There are two main hybrid redundancy classes presented:

1.  **`HybridRedundancy` (cell `wRBeMu4PjOJL`)**:
    *   This is the basic hybrid implementation. It first checks if `model_1` and `model_2` agree. If they do, it returns their common prediction (DMR path). If they disagree, it involves `model_3` and performs a Triple Modular Redundancy (TMR) majority vote among `model_1`, `model_2`, and `model_3`.
    *   It returns only the final prediction.

2.  **`HybridRedundancy_2` (cell `szW27mGEj4Rm`)**:
    *   This is an enhanced version of the hybrid design. It also follows the DMR-then-TMR logic, but it provides additional output: a `usage_flag` for each image processed.
    *   The `usage_flag` indicates whether the decision for that specific image was made by the DMR path (models 1 and 2 agreed, flag = 0) or if it required the TMR path (models 1 and 2 disagreed, and model 3 was consulted, flag = 1).
    *   This `usage_flag` is crucial for detailed analysis, allowing functions like `evaluate_hybrid_model_2` to differentiate and report on the performance and usage of each redundancy path.

In [ ]:
class HybridRedundancy(nn.Module):
    def __init__(self, m1, m2, m3):
        super().__init__()
        self.m1 = m1
        self.m2 = m2
        self.m3 = m3

    def forward(self, x):

        o1 = self.m1(x)
        o2 = self.m2(x)

        b1 = (torch.sigmoid(o1) > 0.5).to(torch.int64)
        b2 = (torch.sigmoid(o2) > 0.5).to(torch.int64)

        if torch.equal(b1, b2):
            return b1  # DMR output

        # TMR run model 3
        o3 = self.m3(x)
        b3 = (torch.sigmoid(o3) > 0.5).to(torch.int64)

        majority = (b1 & b2) | (b1 & b3) | (b2 & b3)

        return majority

In [ ]:
hybrid_model = HybridRedundancy(model_1, model_2, model_3)
hybrid_model.eval()

hybrid_tmr = torch.jit.script(hybrid_model)
hybrid_tmr.save("hybrid_model.pt")

In [ ]:
device = "cpu"
Hybrid_model = torch.jit.load("hybrid_model.pt", map_location=device)
Hybrid_model.eval()
print ("✅") #just to make sure the model can be loaded

In [ ]:
def evaluate_model(model, dataloader, device="cpu", N_list=None):

    model.eval()
    model.to(device)

    if N_list is None:
        N_list = [len(dataloader.dataset)]

    for N in N_list:
        correct = 0
        total = 0
        num_processed = 0

        start_time = time.time()

        with torch.no_grad():
            for images, labels in tqdm(dataloader):
                if N is not None and num_processed >= N:
                    break

                images, labels = images.to(device), labels.to(device)

                if images.dim() == 3:  # single image
                    images = images.unsqueeze(0)

                outputs = model(images)

                # Your original method for prediction
                predicted = outputs.view(-1)
                target_labels = labels.int().view(-1)

                correct += (predicted == target_labels).sum().item()
                total += target_labels.numel()
                num_processed += target_labels.numel()

        end_time = time.time()
        accuracy = 100 * correct / total
        elapsed_time = end_time - start_time

        print(f"\n N = {N} images → Accuracy: {accuracy:.2f}%, Time: {elapsed_time:.2f}s")

In [ ]:
# N_list = [1, 10, 20, 50, 100, 500, 1000] #Colon
N_list = [1, 10, 20, 50, 100, 400, 1000]
evaluate_model(TMR_model, test_loader,device = "cpu", N_list=N_list)

## Hybrid design with flag

In [ ]:
class HybridRedundancy_2(nn.Module):
    def __init__(self, m1, m2, m3):
        super().__init__()
        self.m1 = m1
        self.m2 = m2
        self.m3 = m3

    def forward(self, x):
        # DMR stage
        o1 = self.m1(x)
        o2 = self.m2(x)

        b1 = (torch.sigmoid(o1) > 0.5).to(torch.int64)
        b2 = (torch.sigmoid(o2) > 0.5).to(torch.int64)

        final_pred = b1.clone()
        usage_flag = torch.zeros_like(final_pred)

        for i in range(x.size(0)):
            if b1[i] != b2[i]:

                o3 = self.m3(x[i].unsqueeze(0))
                b3 = (torch.sigmoid(o3) > 0.5).to(torch.int64).squeeze(0)
                # majority vote
                final_pred[i] = (b1[i] & b2[i]) | (b1[i] & b3) | (b2[i] & b3)
                usage_flag[i] = 1  # TMR used

        return final_pred, usage_flag

In [ ]:
hybrid_model_2 = HybridRedundancy_2(model_1, model_2, model_3)
hybrid_model_2.eval()

hybrid_tmr = torch.jit.script(hybrid_model_2)
hybrid_tmr.save("hybrid_model_2.pt")

In [ ]:
device = "cpu"
Hybrid_model_2 = torch.jit.load("hybrid_model_2.pt", map_location=device)
Hybrid_model_2.eval()
print ("✅") #just to make sure the .pt file can be loaded

In [ ]:
def evaluate_hybrid_model_2(model, dataloader, device="cpu", N=None):

    model.eval()
    model.to(device)

    correct = 0
    total = 0
    num_dmr = 0
    num_tmr = 0
    num_processed = 0

    false_dmr = 0
    false_tmr = 0

    dmr_cases = []
    tmr_cases = []

    start_time = time.time()

    dataset = dataloader.dataset   # to access image paths

    with torch.no_grad():
        for images, labels in tqdm(dataloader):

            if N is not None and num_processed >= N:
                break

            images, labels = images.to(device), labels.to(device)

            if images.dim() == 3:
                images = images.unsqueeze(0)

            outputs, usage_flags = model(images)

            predicted = outputs.view(-1)
            target_labels = labels.int().view(-1)
            usage_flags = usage_flags.view(-1)

            # individual model predictions
            o1 = model.m1(images)
            o2 = model.m2(images)

            b1 = (torch.sigmoid(o1) > 0.5).to(torch.int64).view(-1)
            b2 = (torch.sigmoid(o2) > 0.5).to(torch.int64).view(-1)

            correct += (predicted == target_labels).sum().item()
            total += target_labels.numel()

            for i in range(len(predicted)):

                img_index = num_processed + i
                img_path = dataset.samples[img_index][0]

                m1 = int(b1[i].item())
                m2 = int(b2[i].item())
                final_pred = int(predicted[i].item())
                true_label = int(target_labels[i].item())

                # DMR
                if usage_flags[i] == 0:
                    num_dmr += 1

                    if final_pred != true_label:
                        false_dmr += 1
                        dmr_cases.append(
                            f"Index {img_index}: {img_path} \npreds=({m1}, {m2}) | true={true_label} | DMR={final_pred}"
                        )

                # TMR
                else:
                    num_tmr += 1

                    o3 = model.m3(images[i].unsqueeze(0))
                    b3 = int((torch.sigmoid(o3) > 0.5).item())

                    if final_pred != true_label:
                        false_tmr += 1
                        tmr_cases.append(
                            f"Index {img_index}: {img_path} \npreds=({m1}, {m2}, {b3}) | true={true_label} | TMR={final_pred}"
                        )

            num_processed += target_labels.numel()

    end_time = time.time()
    accuracy = 100 * correct / total
    false_count = false_dmr + false_tmr

    print("\n false DMR agreements")
    for case in dmr_cases:
        print(case)

    print("\n false TMR agreements")
    for case in tmr_cases:
        print(case)

    print("\n________________________________")
    print(f"Accuracy: {accuracy:.2f}%")
    print(f"Images processed: {total}")
    print(f"DMR used: {num_dmr}")
    print(f"TMR used: {num_tmr}")
    print(f"False DMR agreements: {false_dmr}")
    print(f"False TMR agreements: {false_tmr}")
    print(f"Total false agreements: {false_count}")
    print("__________________________________")
    return accuracy, num_dmr, num_tmr, false_count

In [ ]:
accuracy, num_dmr, num_tmr, false_count = evaluate_hybrid_model_2(Hybrid_model_2, test_loader, device="cpu", N=1000)

### Additional metrics

To select faulty replica

    faulty_replica:
        0 -> model 1 faulty
        1 -> model 2 faulty
        2 -> model 3 faulty


In [ ]:
def calculate_fault_metrics(
    true_labels,
    final_preds,
    detected_flags,
    recovered_flags,
    system_output_available=None,
):


    true_labels = np.asarray(true_labels, dtype=int)
    final_preds = np.asarray(final_preds, dtype=int)
    detected_flags = np.asarray(detected_flags, dtype=bool)
    recovered_flags = np.asarray(recovered_flags, dtype=bool)

    n = len(true_labels)

    if system_output_available is None:
        system_output_available = np.ones(n, dtype=bool)
    else:
        system_output_available = np.asarray(
            system_output_available, dtype=bool
        )


    valid = system_output_available

    y_true = true_labels[valid]
    y_pred = final_preds[valid]

    TP = np.sum((y_true == 1) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))

    # Accuracy among available outputs
    output_accuracy = (
        (TP + TN) / len(y_true) * 100
        if len(y_true) > 0 else 0
    )

    fpr = (
        FP / (FP + TN) * 100
        if (FP + TN) > 0 else 0
    )

    fnr = (
        FN / (FN + TP) * 100
        if (FN + TP) > 0 else 0
    )

    total_faults = n
    detected = np.sum(detected_flags)

    detection_coverage = (
        detected / total_faults * 100
        if total_faults > 0 else 0
    )


    recovered = np.sum(
        detected_flags & recovered_flags
    )

    recovery_coverage = (
        recovered / detected * 100
        if detected > 0 else 0
    )

    incorrect_final = (
        final_preds != true_labels
    )

    sdc_cases = (
        (~detected_flags)
        & incorrect_final
        & system_output_available
    )

    sdc_count = np.sum(sdc_cases)

    sdc_rate = (
        sdc_count / total_faults * 100
        if total_faults > 0 else 0
    )


    # IMPORTANT:
    # DMR disagreement = no final DMR output.
    # Therefore unresolved DMR samples are NOT counted as correct in end-to-end accuracy.
    correct_final = np.sum(
        system_output_available &
        (final_preds == true_labels)
    )

    end_to_end_accuracy = (
        correct_final / total_faults * 100
        if total_faults > 0 else 0
    )

    print("\n" + "=" * 60)
    print("FAULT-TOLERANCE EVALUATION")
    print("=" * 60)

    print(f"Total fault-injected samples : {total_faults}")

    print("\n--- Fault Detection ---")
    print(f"Detected faults              : {detected}")
    print(f"Fault detection coverage     : {detection_coverage:.2f}%")

    print("\n--- Fault Recovery ( like the masking but we consider the ground truth) ---")
    print(f"Successfully recovered       : {recovered}")
    print(f"Fault recovery coverage      : {recovery_coverage:.2f}%")

    print("\n--- Silent Data Corruption ---")
    print(f"SDC cases                    : {sdc_count}")
    print(f"SDC rate                     : {sdc_rate:.2f}%")

    print("\n--- Diagnostic Errors ---")
    print(f"TP                           : {TP}")
    print(f"TN                           : {TN}")
    print(f"FP                           : {FP}")
    print(f"FN                           : {FN}")
    print(f"False-positive rate          : {fpr:.2f}%")
    print(f"False-negative rate          : {fnr:.2f}%")

    print("\n--- Diagnostic Accuracy ---")
    print(f"Available-output accuracy    : {output_accuracy:.2f}%")
    print(f"End-to-end diagnostic accuracy: {end_to_end_accuracy:.2f}%")

    print("=" * 60)

    return {
        "total_faults": int(total_faults),
        "detected_faults": int(detected),
        "detection_coverage": detection_coverage,
        "recovered_faults": int(recovered),
        "recovery_coverage": recovery_coverage,
        "sdc_count": int(sdc_count),
        "sdc_rate": sdc_rate,
        "TP": int(TP),
        "TN": int(TN),
        "FP": int(FP),
        "FN": int(FN),
        "false_positive_rate": fpr,
        "false_negative_rate": fnr,
        "available_output_accuracy": output_accuracy,
        "end_to_end_accuracy": end_to_end_accuracy,
    }

In [ ]:
def evaluate_DMR_faults(
    dmr_model,
    dataloader,
    faulty_replica=0,
    device="cpu"
):


    dmr_model.eval()
    dmr_model.to(device)

    true_labels = []
    replica_preds = [[], []]
    final_preds = []
    detected_flags = []
    recovered_flags = []
    output_available = []

    with torch.no_grad():

        for images, labels in tqdm(dataloader, desc="Evaluating DMR"):

            images = images.to(device)
            labels = labels.to(device).view(-1).int()

            if images.dim() == 3:
                images = images.unsqueeze(0)

            result = dmr_model(images)

            b1 = result["b1"].view(-1).int()
            b2 = result["b2"].view(-1).int()

            agreement = result["agreement"].view(-1).bool()

            for i in range(len(labels)):

                p1 = int(b1[i].item())
                p2 = int(b2[i].item())
                true = int(labels[i].item())

                replicas = [p1, p2]

                faulty_prediction = replicas[faulty_replica]

                healthy_indices = [
                    j for j in range(2)
                    if j != faulty_replica
                ]

                detected = any(
                    faulty_prediction != replicas[j]
                    for j in healthy_indices
                )

                available = bool(agreement[i].item())

                if available:
                    final_prediction = p1
                    recovered = (
                        detected and
                        final_prediction == true
                    )
                else:
                    final_prediction = 0
                    recovered = False

                true_labels.append(true)

                replica_preds[0].append(p1)
                replica_preds[1].append(p2)

                final_preds.append(final_prediction)
                detected_flags.append(detected)
                recovered_flags.append(recovered)
                output_available.append(available)

    return calculate_fault_metrics(
        true_labels=true_labels,
        final_preds=final_preds,
        detected_flags=detected_flags,
        recovered_flags=recovered_flags,
        system_output_available=output_available
    )

In [ ]:
DMR_model = torch.jit.load(
    "DMR_model.pt",
    map_location="cpu"
)

DMR_results = evaluate_DMR_faults(
    DMR_model,
    test_loader,
    faulty_replica=1,   # change to 1 if model_2 is faulty
    device="cpu"
)

In [ ]:
def evaluate_TMR_faults(
    tmr_model,
    dataloader,
    faulty_replica=0,
    device="cpu"
):


    tmr_model.eval()
    tmr_model.to(device)

    true_labels = []
    replica_preds = [[], [], []]
    final_preds = []
    detected_flags = []
    recovered_flags = []

    with torch.no_grad():

        for images, labels in tqdm(dataloader, desc="Evaluating TMR"):

            images = images.to(device)
            labels = labels.to(device).view(-1).int()

            if images.dim() == 3:
                images = images.unsqueeze(0)

            # Individual replica predictions
            o1 = tmr_model.m1(images)
            o2 = tmr_model.m2(images)
            o3 = tmr_model.m3(images)

            b1 = (torch.sigmoid(o1) > 0.5).to(torch.int64).view(-1)
            b2 = (torch.sigmoid(o2) > 0.5).to(torch.int64).view(-1)
            b3 = (torch.sigmoid(o3) > 0.5).to(torch.int64).view(-1)

            # TMR majority
            tmr_output = (
                (b1 & b2) |
                (b1 & b3) |
                (b2 & b3)
            )

            for i in range(len(labels)):

                preds = [
                    int(b1[i].item()),
                    int(b2[i].item()),
                    int(b3[i].item())
                ]

                true = int(labels[i].item())

                faulty_prediction = preds[faulty_replica]

                healthy_predictions = [
                    preds[j]
                    for j in range(3)
                    if j != faulty_replica
                ]

                detected = any(
                    faulty_prediction != p
                    for p in healthy_predictions
                )

                final_prediction = int(tmr_output[i].item())

                recovered = (
                    detected and
                    final_prediction == true
                )

                true_labels.append(true)

                for j in range(3):
                    replica_preds[j].append(preds[j])

                final_preds.append(final_prediction)
                detected_flags.append(detected)
                recovered_flags.append(recovered)

    return calculate_fault_metrics(
        true_labels=true_labels,
        final_preds=final_preds,
        detected_flags=detected_flags,
        recovered_flags=recovered_flags
    )

In [ ]:
TMR_model = torch.jit.load(
    "TMR_model.pt",
    map_location="cpu"
)

TMR_results = evaluate_TMR_faults(
    TMR_model,
    test_loader,
    faulty_replica=1,
    device="cpu"
)

In [ ]:
def evaluate_Hybrid_faults(
    hybrid_model,
    dataloader,
    faulty_replica=0,
    device="cpu"
):

    hybrid_model.eval()
    hybrid_model.to(device)

    true_labels = []

    replica_preds = [[], [], []]

    final_preds = []
    detected_flags = []
    recovered_flags = []

    dmr_path_count = 0
    tmr_path_count = 0

    with torch.no_grad():

        for images, labels in tqdm(
            dataloader,
            desc="Evaluating Hybrid"
        ):

            images = images.to(device)
            labels = labels.to(device).view(-1).int()

            if images.dim() == 3:
                images = images.unsqueeze(0)

            # Run Hybrid
            final_output, usage_flags = hybrid_model(images)

            final_output = final_output.view(-1).int()
            usage_flags = usage_flags.view(-1).int()

            # Get all individual predictions
            o1 = hybrid_model.m1(images)
            o2 = hybrid_model.m2(images)

            b1 = (
                torch.sigmoid(o1) > 0.5
            ).to(torch.int64).view(-1)

            b2 = (
                torch.sigmoid(o2) > 0.5
            ).to(torch.int64).view(-1)

            # Model 3 is needed for every sample for
            # analysis, even though Hybrid only executes
            # it when disagreement occurs.
            o3 = hybrid_model.m3(images)

            b3 = (
                torch.sigmoid(o3) > 0.5
            ).to(torch.int64).view(-1)

            for i in range(len(labels)):

                preds = [
                    int(b1[i].item()),
                    int(b2[i].item()),
                    int(b3[i].item())
                ]

                true = int(labels[i].item())

                final_prediction = int(
                    final_output[i].item()
                )

                used_tmr = bool(
                    usage_flags[i].item() == 1
                )

                if used_tmr:
                    tmr_path_count += 1
                else:
                    dmr_path_count += 1


                detected = (preds[0] != preds[1])


                recovered = (
                    detected and
                    final_prediction == true
                )

                true_labels.append(true)

                for j in range(3):
                    replica_preds[j].append(preds[j])

                final_preds.append(final_prediction)
                detected_flags.append(detected)
                recovered_flags.append(recovered)

    results = calculate_fault_metrics(
        true_labels=true_labels,
        final_preds=final_preds,
        detected_flags=detected_flags,
        recovered_flags=recovered_flags
    )

    print("\n--- Hybrid Path Usage ---")
    print(f"DMR path used : {dmr_path_count}")
    print(f"TMR path used : {tmr_path_count}")

    results["dmr_path_count"] = dmr_path_count
    results["tmr_path_count"] = tmr_path_count

    return results

In [ ]:
Hybrid_model = torch.jit.load(
    "hybrid_model_2.pt",
    map_location="cpu"
)

Hybrid_results = evaluate_Hybrid_faults(
    Hybrid_model,
    test_loader,
    faulty_replica=1,
    device="cpu"
)